In [ ]:
from collections import defaultdict, deque
from datetime import date
from pathlib import Path
import re

import pandas as pd

# ---- User settings ---------------------------------------------------------
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'code' else Path('.').resolve()
DATA_ROOT = PROJECT_ROOT / 'data'

# Set to None or 'all' to scan every data/* folder.
# Set to a specific folder to scan only that dataset, for example:
#   KRAKEN_INPUT_FOLDER = 'mergedseq6'
#   KRAKEN_INPUT_FOLDER = 'data/mergedseq6'
KRAKEN_INPUT_FOLDER = 'data/mergedseq5'

def resolve_kraken_search_root(folder):
    if folder is None or str(folder).strip().lower() in {'', 'all'}:
        return DATA_ROOT
    folder = Path(folder)
    if folder.is_absolute():
        return folder
    if folder.parts and folder.parts[0] == 'data':
        return PROJECT_ROOT / folder
    return DATA_ROOT / folder

KRAKEN_SEARCH_ROOT = resolve_kraken_search_root(KRAKEN_INPUT_FOLDER)


def output_dataset_label(search_root):
    search_root = Path(search_root)
    if search_root == DATA_ROOT:
        return 'all'
    if search_root.name.startswith('barcode'):
        return search_root.parent.name
    return search_root.name


RUN_DATE = date.today().strftime('%Y%m%d')
KRAKEN_OUTPUT_LABEL = output_dataset_label(KRAKEN_SEARCH_ROOT)
KRAKEN_DB = Path('/mnt/d/kraken2DB/k2_core_nt_2025')
OUTPUT_DIR = DATA_ROOT / 'kraken_parsed'
PRIMERTRACK_DIR = DATA_ROOT / 'primertrack'

# Leave as None to auto-discover the matching primer/UMI best-hit file.
# Set an explicit path if multiple dated files exist and you want one specific run.
PRIMER_UMI_BEST_HITS_OVERRIDE = None

# override changes to the path in PRIMER_UMI_BEST_HITS_OVERRIDE if automatic file searches is not wanted
def find_primer_umi_best_hits(label, primertrack_dir, override=None):
    if override is not None and str(override).strip():
        path = Path(override)
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        return path

    date_pattern = r'\d{8}'
    name_pattern = re.compile(
        rf'^{re.escape(label)}_seqkit_primer_umi_best_hits_{date_pattern}\.txt$'
    )
    matches = sorted(
        path for path in primertrack_dir.glob(f'{label}_seqkit_primer_umi_best_hits_*.txt')
        if name_pattern.match(path.name)
    )

    if not matches:
        return primertrack_dir / f'{label}_seqkit_primer_umi_best_hits_<YYYYMMDD>.txt'
    if len(matches) > 1:
        match_list = '\n'.join(str(path) for path in matches)
        raise ValueError(
            'Multiple primer/UMI best-hit files matched. Keep only the intended file '
            'in the folder or set PRIMER_UMI_BEST_HITS_OVERRIDE explicitly:\n'
            f'{match_list}'
        )
    return matches[0]


PRIMER_UMI_BEST_HITS = find_primer_umi_best_hits(
    KRAKEN_OUTPUT_LABEL,
    PRIMERTRACK_DIR,
    PRIMER_UMI_BEST_HITS_OVERRIDE,
)

# Set either TARGET_TAXID or TARGET_NAME. If TARGET_TAXID is set, it wins.
TARGET_TAXID = 50557 # insecta, keep it always like this for saproxdna.
TARGET_NAME = None  # example: 'Insecta'

# If True, skip .kraken files that do not yet have the matching .report file.
# Keep True while Kraken2 is still running, because .kraken can exist before .report.
REQUIRE_MATCHING_REPORT = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NAMES_DMP = KRAKEN_DB / 'names.dmp'
NODES_DMP = KRAKEN_DB / 'nodes.dmp'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Kraken search root: {KRAKEN_SEARCH_ROOT}')
print(f'Output label: {KRAKEN_OUTPUT_LABEL}')
print(f'Run date: {RUN_DATE}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Primer/UMI best-hit table: {PRIMER_UMI_BEST_HITS}')

Project root: /mnt/c/Users/mftor/Documents/metabarcoding
Data root: /mnt/c/Users/mftor/Documents/metabarcoding/data
Kraken search root: /mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5
Output label: mergedseq5
Run date: 20260630
Output directory: /mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed
Primer/UMI best-hit table: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260630.txt


In [22]:
# Standardise taxonomic labels and remove non alphabetical characters
def taxon_name_to_filename(value):
    value = str(value).strip().lower()
    value = re.sub(r'[^a-z0-9]+', '_', value)
    return value.strip('_') or 'target_taxon'

# Adds metadata to track folder and experiments
# Based on my own very selfish way of organising and naming stuff, which might not be the best
def path_metadata(path):
    path = Path(path)
    parts = path.parts
    barcode_index = next((i for i, part in enumerate(parts) if part.startswith('barcode')), None)

    barcode = parts[barcode_index] if barcode_index is not None else None
    merge_folder = parts[barcode_index - 1] if barcode_index and barcode_index > 0 else None
    barcode_folder = str(Path(*parts[:barcode_index + 1])) if barcode_index is not None else None

    return {
        'merge_folder': merge_folder,
        'barcode': barcode,
        'barcode_folder': barcode_folder,
        'source_file': str(path),
        'source_name': path.name,
    }


def read_names(names_dmp):
    taxid_to_name = {}
    name_to_taxids = defaultdict(list)

    with open(names_dmp, encoding='utf-8') as handle:
        for line in handle:
            fields = [field.strip() for field in line.split('|')]
            taxid = int(fields[0])
            name = fields[1]
            name_class = fields[3]
            if name_class == 'scientific name':
                taxid_to_name[taxid] = name
                # list, since NCBI's names.dmp names can match multiple taxIDs due to homonyms, synonyms, etc
                # but taxid matches exactly one scientific name, no more.
                # kraken2 deals with taxid throughout the whole process, should be fine to use that taxid to retrieve the name of the intended taxon.
                name_to_taxids[name.lower()].append(taxid)

    return taxid_to_name, name_to_taxids


def read_nodes(nodes_dmp):
    taxid_to_parent = {}
    taxid_to_rank = {}
    children_by_parent = defaultdict(list)

    with open(nodes_dmp, encoding='utf-8') as handle:
        for line in handle:
            fields = [field.strip() for field in line.split('|')]
            taxid = int(fields[0])
            parent_taxid = int(fields[1])
            rank = fields[2]
            taxid_to_parent[taxid] = parent_taxid
            taxid_to_rank[taxid] = rank
            if taxid != parent_taxid:
                children_by_parent[parent_taxid].append(taxid)

    return taxid_to_parent, taxid_to_rank, children_by_parent


def resolve_taxid(target_taxid, target_name, taxid_to_name, name_to_taxids):
    if target_taxid is not None:
        target_taxid = int(target_taxid)
        if target_taxid not in taxid_to_name:
            raise ValueError(f'TARGET_TAXID {target_taxid} was not found in names.dmp')
        return target_taxid

    if not target_name:
        raise ValueError('Set TARGET_TAXID or TARGET_NAME')

    matches = name_to_taxids.get(target_name.lower(), [])
    if not matches:
        raise ValueError(f'TARGET_NAME {target_name!r} was not found as a scientific name in names.dmp')
    if len(matches) > 1:
        print(f'TARGET_NAME {target_name!r} matched multiple taxIDs {matches}; using {matches[0]}')
    return matches[0]

# root_taxid is the target taxon
# returns all taxonomic levels lower than the target taxon (children)
# traversal uses nodes.dmp
# children_by_parent is returned by read_nodes()
def descendant_taxids(root_taxid, children_by_parent):
    descendants = set()
    queue = deque([root_taxid])
    while queue:
        # returns root_taxid
        taxid = queue.popleft()
        if taxid in descendants:
            continue
        descendants.add(taxid)
        queue.extend(children_by_parent.get(taxid, []))

    return descendants

LINEAGE_RANKS = ['kingdom', 'order', 'class', 'family', 'genus', 'species']


def lineage_for_taxid(taxid, taxid_to_parent, taxid_to_rank, taxid_to_name):
    lineage = {rank: None for rank in LINEAGE_RANKS}

    if pd.isna(taxid):
        return lineage

    taxid = int(taxid)
    if taxid <= 0 or taxid not in taxid_to_parent:
        return lineage

    seen = set()
    current = taxid

    while current not in seen and current in taxid_to_parent:
        seen.add(current)
        rank = taxid_to_rank.get(current)
        if rank in lineage and lineage[rank] is None:
            lineage[rank] = taxid_to_name.get(current)

        parent = taxid_to_parent.get(current)
        if parent is None or parent == current:
            break
        current = parent

    return lineage


def lineage_table_for_taxids(taxids, taxid_to_parent, taxid_to_rank, taxid_to_name):
    rows = []
    for value in taxids.dropna().unique():
        taxid = int(value)
        if taxid <= 0:
            continue
        row = {'taxonID': taxid}
        row.update(lineage_for_taxid(taxid, taxid_to_parent, taxid_to_rank, taxid_to_name))
        rows.append(row)

    return pd.DataFrame(rows, columns=['taxonID'] + LINEAGE_RANKS)


def kraken_read_map_path(kraken_path):
    combined_input = Path(str(kraken_path)[:-len('.kraken')])
    return combined_input.with_name(f'{combined_input.stem}.read_map.tsv')


def read_kraken_read_map(kraken_path):
    map_path = kraken_read_map_path(kraken_path)
    columns = [
        'merge_folder',
        'barcode',
        'kraken_read_id',
        'record_type',
        'amplicon_sorter_file',
        'amplicon_sorter_group',
        'amplicon_sorter_read_number',
        'ont_read_id',
        'ont_read_length',
        'kraken_sequence_length',
    ]
    output_columns = [
        'source_file',
        'readID',
        'record_type',
        'amplicon_sorter_file',
        'amplicon_sorter_group',
        'amplicon_sorter_read_number',
        'ont_read_id',
        'ont_read_length',
        'kraken_sequence_length',
    ]

    if not map_path.exists():
        print(f'Kraken read map not found, keeping Kraken read IDs only: {map_path}')
        return pd.DataFrame(columns=output_columns)

    read_map = pd.read_csv(map_path, sep='\t', dtype={'kraken_read_id': 'string', 'ont_read_id': 'string'})
    read_map = read_map.rename(columns={'kraken_read_id': 'readID'})
    read_map['source_file'] = str(kraken_path)
    return read_map[output_columns]


def read_primer_umi_best_hits(path):
    path = Path(path)
    read_keys = ['merged_folder', 'barcode_folder', 'seqID']
    primer_fields = [
        'patternName',
        'pattern',
        'strand',
        'start',
        'end',
        'matched',
        'mismatch_count',
        'informative_pattern_length',
        'matched_length',
        'primer_identity',
    ]
    directions = ['forward', 'reverse']
    primer_columns = read_keys + [
        f'{direction}_{field}'
        for direction in directions
        for field in primer_fields
    ]
    output_columns = ['merge_folder', 'barcode', 'ont_read_id'] + [
        f'{direction}_primer_umi_{"best_hit" if field == "patternName" else field.removeprefix("primer_")}'
        for direction in directions
        for field in primer_fields
    ]

    if not path.exists():
        print(f'Primer/UMI best-hit table not found, skipping join: {path}')
        return pd.DataFrame(columns=output_columns)

    primer_hits = pd.read_csv(path, sep='\t', usecols=lambda col: col in primer_columns)
    if primer_hits.empty:
        return pd.DataFrame(columns=output_columns)

    required_directional_columns = {f'{direction}_patternName' for direction in directions}
    missing_directional_columns = required_directional_columns.difference(primer_hits.columns)
    if missing_directional_columns:
        raise ValueError(
            f'Primer/UMI table uses the old single-best-hit format and is missing {sorted(missing_directional_columns)}. '
            'Rerun primer_umi_seqkit_parser.ipynb to generate forward and reverse best hits.'
        )

    rename_columns = {
        'seqID': 'ont_read_id',
        'merged_folder': 'merge_folder',
        'barcode_folder': 'barcode',
    }
    rename_columns.update({
        f'{direction}_{field}': f'{direction}_primer_umi_{"best_hit" if field == "patternName" else field.removeprefix("primer_")}'
        for direction in directions
        for field in primer_fields
    })
    primer_hits = primer_hits.rename(columns=rename_columns)

    primer_hits['ont_read_id'] = primer_hits['ont_read_id'].astype('string')
    return primer_hits[output_columns]


In [23]:
# Load NCBI taxonomy and build the target clade taxID set
taxid_to_name, name_to_taxids = read_names(NAMES_DMP)
taxid_to_parent, taxid_to_rank, children_by_parent = read_nodes(NODES_DMP)

target_taxid = resolve_taxid(TARGET_TAXID, TARGET_NAME, taxid_to_name, name_to_taxids)
target_name = taxid_to_name[target_taxid]
target_rank = taxid_to_rank.get(target_taxid)
target_slug = taxon_name_to_filename(target_name)
target_descendant_taxids = descendant_taxids(target_taxid, children_by_parent)

target_taxonomy = pd.DataFrame({'taxonID': sorted(target_descendant_taxids)})
target_taxonomy['taxonName'] = target_taxonomy['taxonID'].map(taxid_to_name)
target_taxonomy['taxonRank'] = target_taxonomy['taxonID'].map(taxid_to_rank)

taxid_to_taxon_name = taxid_to_name
taxid_to_taxon_rank = taxid_to_rank

print(f'Target: {target_name} (taxID {target_taxid}, rank {target_rank})')
print(f'Target + daughter taxIDs: {len(target_descendant_taxids):,}')
target_taxonomy.head()

Target: Insecta (taxID 50557, rank class)
Target + daughter taxIDs: 799,610


,taxonID,taxonName,taxonRank
0,6826,Euscelis,genus
1,6827,Euscelis plebejus,species
2,6961,Odonata,order
3,6962,Anisoptera,infraorder
4,6964,Libellulidae,family


In [24]:
# Find Kraken2 outputs in the selected search root.
if not KRAKEN_SEARCH_ROOT.exists():
    raise FileNotFoundError(f'Kraken search root does not exist: {KRAKEN_SEARCH_ROOT}')

all_kraken_files = sorted(KRAKEN_SEARCH_ROOT.glob('**/barcode*/**/*.kraken'))
all_report_files = sorted(KRAKEN_SEARCH_ROOT.glob('**/barcode*/**/*.report'))
report_files_set = set(all_report_files)

missing_report_files = sorted(
    Path(str(path)[:-len('.kraken')] + '.report')
    for path in all_kraken_files
    if Path(str(path)[:-len('.kraken')] + '.report') not in report_files_set
)

kraken_files = [
    path for path in all_kraken_files
    if (not REQUIRE_MATCHING_REPORT)
    or Path(str(path)[:-len('.kraken')] + '.report') in report_files_set
]

print(f'Kraken search root: {KRAKEN_SEARCH_ROOT}')
print(f'Kraken files found: {len(all_kraken_files)}')
print(f'Kraken files selected: {len(kraken_files)}')
print(f'Report files found: {len(all_report_files)}')
print(f'Kraken files missing matching report: {len(missing_report_files)}')

for path in missing_report_files[:10]:
    print('missing report:', path)

Kraken search root: /mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5
Kraken files found: 6
Kraken files selected: 5
Report files found: 5
Kraken files missing matching report: 1
missing report: /mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode06/filtered/combined/kraken2_combined_input.fasta.report


In [25]:
def read_kraken_file(path):
    columns = ['status', 'readID', 'taxonID', 'length', 'LCA_list']
    df = pd.read_csv(path, sep='\t', header=None, names=columns, dtype={'readID': 'string', 'taxonID': 'Int64'})
    if df.empty:
        return df

    metadata = path_metadata(path)
    for key, value in metadata.items():
        df[key] = value

    return df


kraken_tables = [read_kraken_file(path) for path in kraken_files]
kraken_reads = (
    pd.concat([df for df in kraken_tables if not df.empty], ignore_index=True)
    if kraken_tables
    else pd.DataFrame(columns=['status', 'readID', 'taxonID', 'length', 'LCA_list'])
)

if not kraken_reads.empty:
    kraken_reads['taxonID'] = pd.to_numeric(kraken_reads['taxonID'], errors='coerce').astype('Int64')
    # Kraken2 assigns taxonID, organism_name/rank are looked up from NCBI names.dmp and nodes.dmp.
    kraken_reads['organism_name'] = kraken_reads['taxonID'].map(taxid_to_name)
    kraken_reads['organism_rank'] = kraken_reads['taxonID'].map(taxid_to_rank)

    lineage_by_taxid = lineage_table_for_taxids(
        kraken_reads['taxonID'],
        taxid_to_parent,
        taxid_to_rank,
        taxid_to_name,
    )
    kraken_reads = kraken_reads.merge(lineage_by_taxid, on='taxonID', how='left')
    kraken_reads['is_target_or_descendant'] = kraken_reads['taxonID'].isin(target_descendant_taxids)
else:
    kraken_reads['organism_name'] = pd.Series(dtype='object')
    kraken_reads['organism_rank'] = pd.Series(dtype='object')
    for rank in LINEAGE_RANKS:
        kraken_reads[rank] = pd.Series(dtype='object')
    kraken_reads['is_target_or_descendant'] = pd.Series(dtype='bool')

read_maps = [read_kraken_read_map(path) for path in kraken_files]
kraken_read_map = (
    pd.concat([df for df in read_maps if not df.empty], ignore_index=True)
    if read_maps
    else pd.DataFrame(columns=['source_file', 'readID', 'ont_read_id'])
)

if not kraken_read_map.empty:
    kraken_reads = kraken_reads.merge(
        kraken_read_map,
        on=['source_file', 'readID'],
        how='left',
        validate='many_to_one',
    )
else:
    for column in [
        'record_type',
        'amplicon_sorter_file',
        'amplicon_sorter_group',
        'amplicon_sorter_read_number',
        'ont_read_id',
        'ont_read_length',
        'kraken_sequence_length',
    ]:
        kraken_reads[column] = pd.NA

primer_umi_hits = read_primer_umi_best_hits(PRIMER_UMI_BEST_HITS)
if not primer_umi_hits.empty:
    kraken_reads = kraken_reads.merge(
        primer_umi_hits,
        on=['merge_folder', 'barcode', 'ont_read_id'],
        how='left',
        validate='many_to_one',
    )
else:
    for column in [
        f'{direction}_primer_umi_{field}'
        for direction in ['forward', 'reverse']
        for field in [
            'best_hit',
            'pattern',
            'strand',
            'start',
            'end',
            'matched',
            'mismatch_count',
            'informative_pattern_length',
            'matched_length',
            'identity',
        ]
    ]:
        kraken_reads[column] = pd.NA

target_reads = kraken_reads[kraken_reads['is_target_or_descendant']].copy()

print(f'Parsed Kraken read rows: {len(kraken_reads):,}')
print(f'Kraken reads with forward primer/UMI best hit: {kraken_reads["forward_primer_umi_best_hit"].notna().sum():,}')
print(f'Kraken reads with reverse primer/UMI best hit: {kraken_reads["reverse_primer_umi_best_hit"].notna().sum():,}')
print(f'{target_name} + daughter read rows: {len(target_reads):,}')
kraken_reads.head()


Parsed Kraken read rows: 1,143,013
Kraken reads with forward primer/UMI best hit: 731,037
Kraken reads with reverse primer/UMI best hit: 681,211
Insecta + daughter read rows: 979,676


,status,readID,taxonID,length,LCA_list,merge_folder,barcode,barcode_folder,source_file,source_name,...,reverse_primer_umi_best_hit,reverse_primer_umi_pattern,reverse_primer_umi_strand,reverse_primer_umi_start,reverse_primer_umi_end,reverse_primer_umi_matched,reverse_primer_umi_mismatch_count,reverse_primer_umi_informative_pattern_length,reverse_primer_umi_matched_length,reverse_primer_umi_identity
0,U,64344,0,371,0:337,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,kraken2_combined_input.fasta.kraken,...,reverse_tagged_primer_2,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,-,340.0,386.0,ggtagatatatgcacgctctataaacttcaggatgtccaaaaaatca,0.0,47.0,47.0,1.0
1,U,64547,0,371,0:337,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,kraken2_combined_input.fasta.kraken,...,reverse_tagged_primer_2,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,-,340.0,386.0,ggtagatatatgcacgctctataaacttctggatgtccaaaaaatca,0.0,47.0,47.0,1.0
2,U,68205,0,371,0:337,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,kraken2_combined_input.fasta.kraken,...,reverse_tagged_primer_2,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,-,340.0,386.0,ggtagatatatgcacgctctataaacttcagggtgtccaaaaaatca,0.0,47.0,47.0,1.0
3,U,68405,0,370,0:322 888845:2 0:12,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,kraken2_combined_input.fasta.kraken,...,reverse_tagged_primer_2,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,-,339.0,385.0,ggtagatatatgcacgctctataaacttcagggtgaccaaaaaatca,0.0,47.0,47.0,1.0
4,U,92673,0,370,0:336,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,kraken2_combined_input.fasta.kraken,...,reverse_tagged_primer_2,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,-,339.0,385.0,ggtagatatatgcacgctctataaacttcagggtgtccaaaaaatca,0.0,47.0,47.0,1.0


In [26]:
# Save complete read table plus filtered target-clade table.
all_reads_path = OUTPUT_DIR / f'{KRAKEN_OUTPUT_LABEL}_all_kraken_reads_{RUN_DATE}.csv'
target_reads_path = OUTPUT_DIR / f'{KRAKEN_OUTPUT_LABEL}_{target_slug}_kraken_reads_{RUN_DATE}.csv'
target_taxids_path = OUTPUT_DIR / f'{KRAKEN_OUTPUT_LABEL}_{target_slug}_taxids_{RUN_DATE}.csv'
missing_reports_path = OUTPUT_DIR / f'{KRAKEN_OUTPUT_LABEL}_kraken_files_missing_reports_{RUN_DATE}.csv'

kraken_reads.to_csv(all_reads_path, sep='\t', index=False)
target_reads.to_csv(target_reads_path, sep='\t', index=False)
target_taxonomy.to_csv(target_taxids_path, sep='\t', index=False)
pd.DataFrame({'missing_report_file': [str(path) for path in missing_report_files]}).to_csv(
    missing_reports_path, sep='\t', index=False
)

print('Saved:')
print(all_reads_path)
print(target_reads_path)
print(target_taxids_path)
print(missing_reports_path)

Saved:
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/mergedseq5_all_kraken_reads_20260630.csv
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/mergedseq5_insecta_kraken_reads_20260630.csv
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/mergedseq5_insecta_taxids_20260630.csv
/mnt/c/Users/mftor/Documents/metabarcoding/data/kraken_parsed/mergedseq5_kraken_files_missing_reports_20260630.csv


In [27]:
# Optional summaries for quick inspection.
lineage_columns = ['kingdom', 'order', 'class', 'family', 'genus', 'species']
taxon_lineage = (
    kraken_reads[['taxonID'] + lineage_columns]
    .dropna(subset=['taxonID'])
    .drop_duplicates('taxonID')
)

target_read_counts_by_taxon = (
    target_reads
    .groupby(['taxonID', 'organism_name', 'organism_rank'], dropna=False)
    .size()
    .reset_index(name='read_count')
    .merge(taxon_lineage, on='taxonID', how='left')
    .sort_values('read_count', ascending=False)
)

target_read_counts_by_barcode = (
    target_reads
    .groupby(['merge_folder', 'barcode', 'taxonID', 'organism_name', 'organism_rank'], dropna=False)
    .size()
    .reset_index(name='read_count')
    .merge(taxon_lineage, on='taxonID', how='left')
    .sort_values(['merge_folder', 'barcode', 'read_count'], ascending=[True, True, False])
)

barcode_column_order = [
    'merge_folder',
    'barcode',
    'taxonID',
    'organism_name',
    'organism_rank',
    'kingdom',
    'order',
    'class',
    'family',
    'genus',
    'species',
    'read_count',
]
target_read_counts_by_barcode = target_read_counts_by_barcode[barcode_column_order]

target_read_counts_by_taxon.to_csv(
    OUTPUT_DIR / f'{KRAKEN_OUTPUT_LABEL}_{target_slug}_read_counts_by_taxon_{RUN_DATE}.csv',
    sep='\t', index=False,
)
target_read_counts_by_barcode.to_csv(
    OUTPUT_DIR / f'{KRAKEN_OUTPUT_LABEL}_{target_slug}_read_counts_by_barcode_{RUN_DATE}.csv',
    sep='\t', index=False,
)

print(f'Observed target/daughter taxIDs: {target_read_counts_by_taxon.shape[0]:,}')
target_read_counts_by_taxon.head(25)


Observed target/daughter taxIDs: 10,569


,taxonID,organism_name,organism_rank,read_count,kingdom,order,class,family,genus,species
6729,1888014,Pseudolycoriella brunnea,species,53875,Metazoa,Diptera,Insecta,Sciaridae,Pseudolycoriella,Pseudolycoriella brunnea
1644,265459,unclassified Sciaridae,no rank,51353,Metazoa,Diptera,Insecta,Sciaridae,None,None
6593,1881783,Orthocladiinae sp. BOLD-2016,species,41347,Metazoa,Diptera,Insecta,Chironomidae,None,Orthocladiinae sp. BOLD-2016
6346,1817649,Lycoriella brevipila,species,40591,Metazoa,Diptera,Insecta,Sciaridae,Lycoriella,Lycoriella brevipila
9907,2909299,Sciaridae sp.,species,38337,Metazoa,Diptera,Insecta,Sciaridae,None,Sciaridae sp.
77,7184,Sciaridae,family,27289,Metazoa,Diptera,Insecta,Sciaridae,None,None
9854,2890712,Scatopsciara brevicornis,species,26505,Metazoa,Diptera,Insecta,Sciaridae,Scatopsciara,Scatopsciara brevicornis
6058,1783334,Bradysia placida,species,20103,Metazoa,Diptera,Insecta,Sciaridae,Bradysia,Bradysia placida
9761,2882605,Phaonia mystica,species,17252,Metazoa,Diptera,Insecta,Muscidae,Phaonia,Phaonia mystica
6331,1817620,Corynoptera deserta,species,15257,Metazoa,Diptera,Insecta,Sciaridae,Corynoptera,Corynoptera deserta
